# Phase 2: Pipeline Machine Learning (S2-3)

## Objectifs:
- Nettoyage des données: doublons, valeurs manquantes, outliers
- Feature engineering: 5 nouvelles features (hour, day_of_week, month, mean_temp, delta_temp)
- Split train/test & normalisation (StandardScaler)
- Modélisation ML: Random Forest & XGBoost
- Optimisation hyperparamètres (GridSearchCV) & tracking MLflow
- Évaluation des performances (MAE, RMSE, R²)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import joblib
import os

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Chargement et Inspection des Données

In [ ]:
# Load data
df = pd.read_csv('../data/energydata_complete.csv')
print('Shape:', df.shape)
print('\nMissing values:\n', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())
print('\nTarget stats:\n', df['Appliances'].describe())

# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['Appliances'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Appliances Distribution')
sns.boxplot(y=df['Appliances'], ax=axes[1])
axes[1].set_title('Appliances Boxplot')
plt.tight_layout()
plt.show()

## 2. Nettoyage des Données

In [ ]:
# Remove duplicates (if any)
initial_shape = df.shape
df = df.drop_duplicates()
print(f'Duplicates removed: {initial_shape[0] - df.shape[0]}')

# Remove missing (if any)
df = df.dropna()
print(f'Missing removed: {initial_shape[0] - df.shape[0]}')

# Outlier removal using IQR on target
Q1 = df['Appliances'].quantile(0.25)
Q3 = df['Appliances'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers_before = ((df['Appliances'] < lower_bound) | (df['Appliances'] > upper_bound)).sum()

df_clean = df[(df['Appliances'] >= lower_bound) & (df['Appliances'] <= upper_bound)].copy()
print(f'Outliers removed: {outliers_before} -> Shape: {df_clean.shape}')

# Visualize outlier removal
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(y=df['Appliances'], ax=axes[0])
axes[0].set_title('Before Outlier Removal')
sns.boxplot(y=df_clean['Appliances'], ax=axes[1])
axes[1].set_title('After Outlier Removal')
plt.tight_layout()
plt.show()

df = df_clean

## 3. Feature Engineering (5 nouvelles features)

In [ ]:
# Time features
df['hour'] = (df['NSM'] // 60).astype(int)
df['day_of_week'] = df['Day_of_week']
# Month derivation from Week (approx 20 weeks data ~5 months)
df['month'] = np.clip(((df['Week'] - 1) // 4) + 1, 1, 12).astype(int)

# Temperature features
temp_cols = [col for col in df.columns if col.startswith('T')]
df['mean_temp'] = df[temp_cols].mean(axis=1)
df['delta_temp'] = df['T1'] - df['T_out']

print('New features created: hour, day_of_week, month, mean_temp, delta_temp')
print('Shape after FE:', df.shape)

## 4. Sélection des Features et Split Train/Test + Normalisation

In [ ]:
# Feature columns (original + new, drop irrelevant)
feature_cols = [
    'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5',
    'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg',
    'RH_out', 'Windspeed', 'Visibility', 'Tdew', 'hour', 'day_of_week', 'month',
    'mean_temp', 'delta_temp'
]
X = df[feature_cols]
y = df['Appliances']

print('X shape:', X.shape)
print('y shape:', y.shape)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

# StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')
print('Scaler saved to models/scaler.pkl')

## 5. Modélisation ML avec MLflow Tracking

In [ ]:
mlflow.set_experiment("Energy_Prediction_Phase2")

with mlflow.start_run(run_name="Phase2_Complete_Pipeline"):
    
    # Baseline: mean prediction
    baseline_pred_train = np.full_like(y_train, y_train.mean())
    baseline_pred_test = np.full_like(y_test, y_train.mean())
    baseline_mae = mean_absolute_error(y_test, baseline_pred_test)
    baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred_test))
    baseline_r2 = r2_score(y_test, baseline_pred_test)
    
    print('Baseline - MAE:', baseline_mae, 'RMSE:', baseline_rmse, 'R²:', baseline_r2)
    mlflow.log_metrics({
        'baseline_mae': baseline_mae,
        'baseline_rmse': baseline_rmse,
        'baseline_r2': baseline_r2
    })
    
    # Random Forest avec GridSearchCV
    rf = RandomForestRegressor(random_state=42, n_jobs=-1)
    rf_param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5]
    }
    rf_grid = GridSearchCV(rf, rf_param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
    rf_grid.fit(X_train_scaled, y_train)
    
    rf_pred = rf_grid.best_estimator_.predict(X_test_scaled)
    rf_mae = mean_absolute_error(y_test, rf_pred)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
    rf_r2 = r2_score(y_test, rf_pred)
    
    print(f'RF Best params: {rf_grid.best_params_}')
    print('RF - MAE:', rf_mae, 'RMSE:', rf_rmse, 'R²:', rf_r2)
    mlflow.log_metrics({'rf_mae': rf_mae, 'rf_rmse': rf_rmse, 'rf_r2': rf_r2})
    mlflow.log_params(rf_grid.best_params_)
    mlflow.sklearn.log_model(rf_grid.best_estimator_, "rf_model")
    joblib.dump(rf_grid.best_estimator_, '../models/rf_model.pkl')
    
    # XGBoost avec GridSearchCV
    xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)
    xgb_param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 6],
        'learning_rate': [0.01, 0.1]
    }
    xgb_grid = GridSearchCV(xgb_model, xgb_param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
    xgb_grid.fit(X_train_scaled, y_train)
    
    xgb_pred = xgb_grid.best_estimator_.predict(X_test_scaled)
    xgb_mae = mean_absolute_error(y_test, xgb_pred)
    xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
    xgb_r2 = r2_score(y_test, xgb_pred)
    
    print(f'XGB Best params: {xgb_grid.best_params_}')
    print('XGB - MAE:', xgb_mae, 'RMSE:', xgb_rmse, 'R²:', xgb_r2)
    mlflow.log_metrics({'xgb_mae': xgb_mae, 'xgb_rmse': xgb_rmse, 'xgb_r2': xgb_r2})
    mlflow.log_params(xgb_grid.best_params_)
    mlflow.xgboost.log_model(xgb_grid.best_estimator_, "xgb_model")
    joblib.dump(xgb_grid.best_estimator_, '../models/xgb_model.pkl')
    
    # Save best model (lowest RMSE)
    models_metrics = {
        'baseline': baseline_rmse,
        'rf': rf_rmse,
        'xgb': xgb_rmse
    }
    best_model_name = min(models_metrics, key=models_metrics.get)
    if best_model_name == 'rf':
        best_model = rf_grid.best_estimator_
    else:
        best_model = xgb_grid.best_estimator_
    joblib.dump(best_model, '../models/best_model.pkl')
    mlflow.log_param('best_model', best_model_name)
    print(f'Best model (min RMSE): {best_model_name} saved')

## 6. Évaluation des Performances

In [ ]:
# Results table
results = pd.DataFrame({
    'Model': ['Baseline', 'Random Forest', 'XGBoost'],
    'MAE': [baseline_mae, rf_mae, xgb_mae],
    'RMSE': [baseline_rmse, rf_rmse, xgb_rmse],
    'R²': [baseline_r2, rf_r2, xgb_r2]
})
print('Performance Comparison:')
print(results.round(4))

# Predictions plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = {'RF': rf_pred, 'XGB': xgb_pred}
for i, (name, pred) in enumerate(models.items()):
    axes[i].scatter(y_test, pred, alpha=0.6)
    min_val, max_val = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
    axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
    axes[i].set_xlabel('Actual')
    axes[i].set_ylabel('Predicted')
    axes[i].set_title(f'{name} (R²={r2_score(y_test, pred):.3f})')

residuals_rf = y_test - rf_pred
axes[2].hist(residuals_rf, bins=50, alpha=0.7)
axes[2].set_xlabel('Residuals')
axes[2].set_ylabel('Frequency')
axes[2].set_title('RF Residuals Distribution')
plt.tight_layout()
plt.show()

print('\n✅ Pipeline Phase 2 terminée!')
print('📁 Fichiers sauvegardés: models/{scaler,rf,xgb,best}_model.pkl')
print('🔗 MLflow: mlflow ui (port 5000)')